# EF-02 | NBS CPI Data Parser
Parses the downloaded NBS `.xls` files and produces a clean `nbs_cpi_monthly.csv`.

**What we know from inspecting the files:**
- Each file has multiple sheets — one per year (e.g. `2022_REBASED SERIES`)
- We want the sheet matching the file's year
- Row index 2 = `INFLATION RATE` (year-on-year %)
- Row index 3 = `ALL ITEMS INDEX` (CPI index value)
- Columns 3 onwards = monthly values (Jan → Dec)
- Legacy `.xls` files require `engine='xlrd'`

## Cell 1 — Imports

In [1]:
import os
import re
import pandas as pd

# Folder where your downloaded files live
DATA_DIR   = 'nbs_cpi_raw'    # change if yours is different
OUTPUT_CSV = 'nbs_cpi_monthly.csv'
TARGET_YEARS = [2022, 2023, 2024]

# Row positions inside each sheet (0-indexed, confirmed from inspection)
ROW_INFLATION = 2    # INFLATION RATE row
ROW_CPI_INDEX = 3    # ALL ITEMS INDEX row
COL_DATA_START = 3   # first column with actual monthly data

print('Config ready.')

Config ready.


## Cell 2 — Find the Right Sheet

Each file contains sheets for multiple years. We want the sheet
whose name contains the target year and `REBASED` (the newer series).

In [2]:
def find_target_sheet(xls_path, year):
    """
    Opens the xls and finds the sheet matching the target year.
    Prefers sheets with 'REBASED' in the name (newer methodology).
    Returns the sheet name or None if not found.
    """
    try:
        xl = pd.ExcelFile(xls_path, engine='xlrd')
        sheets = xl.sheet_names
    except Exception as e:
        print(f'  Cannot open {os.path.basename(xls_path)}: {e}')
        return None

    year_str = str(year)

    # First preference: sheet with year + REBASED (e.g. '2022_REBASED SERIES')
    for s in sheets:
        if year_str in s and 'REBASED' in s.upper():
            return s

    # Second preference: any sheet containing the year
    for s in sheets:
        if year_str in s:
            return s

    print(f'  No sheet found for year {year} in {os.path.basename(xls_path)}')
    print(f'  Available sheets: {sheets}')
    return None


# Quick test on your April 2022 file
test_files = []
for yr in TARGET_YEARS:
    yr_dir = os.path.join(DATA_DIR, str(yr))
    if os.path.exists(yr_dir):
        files = sorted([f for f in os.listdir(yr_dir) if f.endswith('.xls') or f.endswith('.xlsx')])
        if files:
            test_files.append((yr, os.path.join(yr_dir, files[0])))

print('Sheet detection test:')
for yr, fp in test_files:
    sheet = find_target_sheet(fp, yr)
    print(f'  Year {yr} → Sheet: "{sheet}"  ({os.path.basename(fp)})')

Sheet detection test:


## Cell 3 — Extract Month-Year from Filename

NBS filenames follow patterns like:
- `CPI_Summary_042022.xls` → `2022-04`
- `CPI Summary_122022.xls` → `2022-12`

In [3]:
def extract_year_month(filename):
    """
    Extracts year_month string from NBS CPI filename.
    Returns e.g. '2022-04' or None if not parseable.
    """
    name = os.path.basename(filename)

    # Pattern: MMYYYY  e.g. 042022 or 122023
    m = re.search(r'[_\s](\d{2})(\d{4})\.xls', name, re.IGNORECASE)
    if m:
        month, year = m.group(1), m.group(2)
        if 1 <= int(month) <= 12:
            return f'{year}-{month}'

    # Pattern: month name + year  e.g. 'January 2023'
    month_map = {
        'january':'01','february':'02','march':'03','april':'04',
        'may':'05','june':'06','july':'07','august':'08',
        'september':'09','october':'10','november':'11','december':'12'
    }
    m = re.search(
        r'(january|february|march|april|may|june|july|august|'
        r'september|october|november|december)[_\s]+(\d{4})',
        name, re.IGNORECASE
    )
    if m:
        return f'{m.group(2)}-{month_map[m.group(1).lower()]}'

    return None


# Test it
test_names = [
    'en-1705504571-CPI_Summary_042022.xls',
    'en-1705504571-CPI_Summary_122022.xls',
    'CPI Summary_November 2023.xls',
]
for name in test_names:
    print(f'  {name}  →  {extract_year_month(name)}')

  en-1705504571-CPI_Summary_042022.xls  →  2022-04
  en-1705504571-CPI_Summary_122022.xls  →  2022-12
  CPI Summary_November 2023.xls  →  2023-11


## Cell 4 — Parse One File

Reads one `.xls`, finds the correct sheet, extracts the monthly
CPI index and inflation rate for the file's specific month.

In [4]:
def parse_cpi_file(filepath, year):
    """
    Parses one NBS CPI summary .xls file.
    Returns a single dict: {year_month, cpi_index, inflation_rate}
    or None on failure.
    """
    year_month = extract_year_month(filepath)
    if not year_month:
        print(f'  Could not parse date from: {os.path.basename(filepath)}')
        return None

    target_month = int(year_month.split('-')[1])  # 1-12

    # Find the right sheet
    sheet_name = find_target_sheet(filepath, year)
    if not sheet_name:
        return None

    try:
        df = pd.read_excel(
            filepath,
            sheet_name=sheet_name,
            header=None,
            engine='xlrd'
        )
    except Exception as e:
        print(f'  Read error on {os.path.basename(filepath)}: {e}')
        return None

    # Row 2 = inflation rate, Row 3 = all items index
    # Columns 3+ are monthly values — find the one matching our month
    inflation_row = df.iloc[ROW_INFLATION]
    cpi_row       = df.iloc[ROW_CPI_INDEX]

    # Scan columns from COL_DATA_START to find our target month
    cpi_val       = None
    inflation_val = None

    for col_idx in range(COL_DATA_START, len(df.columns)):
        col_header = df.iloc[1, col_idx]  # Row 1 has the date headers

        # col_header is either a datetime or a string like '2022-04-01'
        try:
            if pd.isna(col_header):
                continue
            col_dt = pd.to_datetime(col_header)
            if col_dt.year == year and col_dt.month == target_month:
                cpi_val       = cpi_row.iloc[col_idx]
                inflation_val = inflation_row.iloc[col_idx]
                break
        except Exception:
            continue

    if cpi_val is None:
        print(f'  Month {target_month}/{year} not found in {os.path.basename(filepath)}')
        return None

    # Convert to float safely
    try:
        cpi_val       = float(cpi_val)
        inflation_val = float(inflation_val) if pd.notna(inflation_val) else None
    except (ValueError, TypeError):
        print(f'  Non-numeric values in {os.path.basename(filepath)}')
        return None

    return {
        'year_month':    year_month,
        'cpi_index':     round(cpi_val, 4),
        'inflation_rate': round(inflation_val, 4) if inflation_val else None,
        'source':        'NBS Tanzania'
    }


# Test on first available file
if test_files:
    yr, fp = test_files[0]
    print(f'Testing parser on: {os.path.basename(fp)}')
    result = parse_cpi_file(fp, yr)
    print(f'Result: {result}')

## Cell 5 — Run on All Files

In [6]:
def parse_all_files(data_dir, target_years):
    records = []
    failed  = []

    for year in target_years:
        yr_dir = os.path.join(data_dir, str(year))
        if not os.path.exists(yr_dir):
            print(f'Folder not found: {yr_dir}')
            continue

        files = sorted([
            f for f in os.listdir(yr_dir)
            if f.lower().endswith('.xls') or f.lower().endswith('.xlsx')
        ])

        print(f'\nYear {year}: {len(files)} files')
        print('-' * 40)

        for fname in files:
            fpath  = os.path.join(yr_dir, fname)
            record = parse_cpi_file(fpath, year)

            if record:
                records.append(record)
                print(f'  ✓ {record["year_month"]}  CPI: {record["cpi_index"]}  Inflation: {record["inflation_rate"]}%')
            else:
                failed.append(fname)

    print(f'\n{"="*50}')
    print(f'Parsed:  {len(records)} monthly records')
    print(f'Failed:  {len(failed)} files')
    if failed:
        print('Failed files:')
        for f in failed:
            print(f'  {f}')

    return records


all_records = parse_all_files(DATA_DIR, TARGET_YEARS)


Year 2022: 9 files
----------------------------------------
  Cannot open en-1705504571-CPI Summary_042022.xls: Missing optional dependency 'xlrd'. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install xlrd.
  Cannot open en-1705504571-CPI Summary_082022.xls: Missing optional dependency 'xlrd'. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install xlrd.
  Cannot open en-1705504571-CPI Summary_102022.xls: Missing optional dependency 'xlrd'. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install xlrd.
  Cannot open en-1705504571-CPI Summary_122022.xls: Missing optional dependency 'xlrd'. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install xlrd.
  Cannot open en-1705504571-CPI_Summary_012022.xls: Missing optional dependency 'xlrd'. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install xlrd.
  Cannot open en-1705504571-CPI_Summary_022022.xls: Missing optional dependency 'xlrd'. Install xlrd >= 2.0.1

## Cell 6 — Build Clean DataFrame & Save

In [ ]:
df_cpi = (
    pd.DataFrame(all_records)
    .drop_duplicates(subset='year_month')
    .sort_values('year_month')
    .reset_index(drop=True)
)

# Sanity check — should have 36 rows for 2022-2024
print(f'Total rows: {len(df_cpi)}  (expected 36 for 3 full years)')
print(f'Date range: {df_cpi["year_month"].min()} → {df_cpi["year_month"].max()}')
print(f'Missing months: {36 - len(df_cpi)}')

df_cpi.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved to {OUTPUT_CSV}')
df_cpi

## Cell 7 — Spot Check Missing Months

If you have fewer than 36 rows, this cell shows which months are missing.

In [ ]:
import itertools

expected = [
    f'{yr}-{mo:02d}'
    for yr, mo in itertools.product(TARGET_YEARS, range(1, 13))
]

got     = set(df_cpi['year_month'].tolist())
missing = [m for m in expected if m not in got]

if missing:
    print(f'Missing {len(missing)} months:')
    for m in missing:
        print(f'  {m}')
else:
    print('All 36 months present. Ready for correlation analysis.')